# InfiniTune Colab v2: Real-Time LLM Fine-Tuning & Evaluation

This notebook runs the **real InfiniTune streaming architecture** on Google Colab GPUs (T4, L4, or A100).

### Supported Deployment Modes:
1. **Mode A (All-in-Colab Full Evaluation)**: Runs native `producer.py`, `trainer.py`, `inference.py`, Apache Kafka 3.9 (KRaft mode), and benchmark suites (`streaming_quality`, `propagation`, `eval_speed`, `claims`) directly on GPU.
2. **Mode B (Split Deployment via Managed Kafka)**: Colab GPU runs the Producer and Trainer, pushing LoRA weight updates via SASL_SSL to managed Kafka (Confluent, Upstash, Redpanda, Aiven). Your local Windows laptop runs `inference.py --listen-updates` to serve live traffic and hot-swap weights without dropping requests.
3. **Mode C (Split Deployment via Google Drive Sync)**: Colab GPU trains and saves atomic checkpoints (`COMMITTED` + `manifest.json` + `safetensors`) directly to Google Drive. Your local laptop runs `inference.py --watch-checkpoints` to verify SHA-256 and hot-swap weights in real-time with zero external networking configuration.

### Cell 1: Environment & GPU Hardware Check
Inspect the allocated NVIDIA GPU, compute capability, and establish the precision policy (`bf16` on Ampere/Ada/Hopper vs `fp16` on Turing T4).

In [ ]:
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU available. In Colab, go to Runtime -> Change runtime type -> T4 / L4 / A100 GPU.")

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"Active GPU: {device_name}")
print(f"Compute Capability: {capability[0]}.{capability[1]}")

# Ampere+ (SM 8.0+) natively supports bf16; Turing (T4, SM 7.5) uses fp16
recommended_precision = "bf16" if capability[0] >= 8 else "fp16"
print(f"Recommended precision policy: {recommended_precision}")

### Cell 2: Mount Google Drive for Persistent Storage
Mount Google Drive under `/content/drive` to persist checkpoints, benchmark results, Hugging Face model caches, and logs across session disconnects.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/infinitune")
PERSIST.mkdir(parents=True, exist_ok=True)
(PERSIST / "runs").mkdir(parents=True, exist_ok=True)
(PERSIST / "checkpoints").mkdir(parents=True, exist_ok=True)
print(f"Persistent storage directory: {PERSIST}")

### Cell 3: Clone Repository at Pinned Branch / Commit
Clone the official InfiniTune repository and checkout the evaluation branch `colab-evaluation-changes-rs`.

In [ ]:
import subprocess
import os

try:
    from google.colab import userdata
    gh_token = userdata.get("GH_TOKEN")
except Exception:
    gh_token = None

if gh_token:
    REPO_URL = f"https://{gh_token}@github.com/rohang1411/Infinitune-Realtime-LLM-Fine-Tuning-Framework.git"
else:
    REPO_URL = "https://github.com/rohang1411/Infinitune-Realtime-LLM-Fine-Tuning-Framework.git"

REPO_DIR = "/content/Infinitune-Realtime-LLM-Fine-Tuning-Framework"
BRANCH = "colab-evaluation-changes-rs"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH} || true

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
print(f"InfiniTune Repository checked out at commit: {commit_hash} on branch {BRANCH}")

### Cell 4: Install Colab Dependencies & Run Environment Census
Install dependencies from `requirements-colab.txt` (preserving Colab's preinstalled CUDA-enabled PyTorch, NumPy, and scikit-learn), followed by running `benchmarks/env_check.py`.

In [ ]:
!pip install -r requirements-colab.txt
!python -m benchmarks.env_check

### Cell 5: Start Apache Kafka (KRaft mode)
Execute `scripts/colab/start_kafka.sh` to install headless JRE, download Kafka 3.9.0, format KRaft storage under `/content/kafka-data`, start the broker, and verify readiness on `localhost:9092`.

In [ ]:
!bash scripts/colab/start_kafka.sh

### Cell 6: Configure Hugging Face Cache & Optional Secrets
Set `HF_HOME` to persistent Drive storage so base model weights are downloaded only once. Safely retrieve optional Hugging Face tokens from Colab Secrets.

In [ ]:
import os
os.environ["HF_HOME"] = str(PERSIST / "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print(f"Hugging Face Cache: {os.environ['HF_HOME']}")

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN successfully loaded from Colab Secrets.")
except Exception:
    print("HF_TOKEN secret not set (public model downloads will still work).")

### Cell 7: Mode A - Run End-to-End Benchmark Suite on Colab GPU
Execute reproducible benchmark suites via `benchmarks.run`. Manifests and raw trial logs are written directly to persistent storage on Drive.

In [ ]:
import time

RUN_NAME = f"colab_run_{int(time.time())}"
RUN_DIR = PERSIST / "runs" / RUN_NAME

# Example: Run streaming_quality suite with GPU acceleration
!python -m benchmarks.run run \
    --suites streaming_quality \
    --profile standard \
    --model distilgpt2 \
    --live-config configs/imdb_quantitative.yaml \
    --device cuda \
    --kafka-bootstrap localhost:9092 \
    --run-dir {RUN_DIR}

### Cell 8: Mode A - Evaluate Stronger Models (Qwen2.5-3B GSM8K / Alpaca)
Train and evaluate larger LLMs using Colab GPU acceleration with automatic `bf16` precision and gradient accumulation.

In [ ]:
QWEN_RUN_NAME = f"qwen3b_gsm8k_{int(time.time())}"
QWEN_RUN_DIR = PERSIST / "runs" / QWEN_RUN_NAME

# Train and evaluate Qwen2.5-3B on GSM8K with GPU acceleration
!python -m benchmarks.run run \
    --suites streaming_quality propagation \
    --profile standard \
    --model Qwen/Qwen2.5-3B \
    --live-config configs/gsm8k_quantitative.yaml \
    --config-overlay configs/overlays/colab_gpu.yaml \
    --device cuda \
    --kafka-bootstrap localhost:9092 \
    --run-dir {QWEN_RUN_DIR}

### Cell 9: Mode B - Split Deployment (Colab GPU Trains -> Managed Kafka -> Laptop Serves)
Offload heavy GPU training to Google Colab while serving live inference on your local Windows laptop via Managed Kafka (SASL_SSL).

In [ ]:
# Load managed Kafka credentials from Colab Secrets
try:
    from google.colab import userdata
    os.environ["INFINITUNE_KAFKA_BOOTSTRAP"] = userdata.get("INFINITUNE_KAFKA_BOOTSTRAP")
    os.environ["INFINITUNE_KAFKA_USERNAME"] = userdata.get("INFINITUNE_KAFKA_USERNAME")
    os.environ["INFINITUNE_KAFKA_PASSWORD"] = userdata.get("INFINITUNE_KAFKA_PASSWORD")
    os.environ["INFINITUNE_ADAPTER_HMAC_KEY"] = userdata.get("INFINITUNE_ADAPTER_HMAC_KEY")
    print("Managed Kafka credentials successfully loaded from Colab Secrets.")
except Exception as e:
    print(f"Note: Set INFINITUNE_KAFKA_* in Colab Secrets to enable managed Kafka: {e}")

print("\n--- Instructions for Local Laptop Serving ---")
print("1. On local laptop, set identical environment variables:")
print("   $env:INFINITUNE_KAFKA_BOOTSTRAP = '<your-broker>:9092'")
print("   $env:INFINITUNE_KAFKA_USERNAME = '<api-key>'")
print("   $env:INFINITUNE_KAFKA_PASSWORD = '<api-secret>'")
print("   $env:INFINITUNE_ADAPTER_HMAC_KEY = '<hmac-key>'")
print("2. Start local inference server on CPU:")
print("   python inference.py --config configs/imdb_quantitative.yaml --config-overlay configs/overlays/managed_kafka.yaml --listen-updates --device cpu")

# 3. Run training on Colab GPU with managed Kafka overlay:
# !python trainer.py --config configs/imdb_quantitative.yaml --config-overlay configs/overlays/managed_kafka.yaml --device cuda

### Cell 10: Mode C - Split Deployment (Colab GPU Trains -> Google Drive Sync -> Laptop Serves)
Zero-network setup: Colab GPU saves atomic checkpoints to Google Drive, and your local laptop automatically detects, verifies, and hot-swaps weights.

In [ ]:
DRIVE_CKPT_DIR = str(PERSIST / "checkpoints")
print(f"Colab Checkpoint Output Directory: {DRIVE_CKPT_DIR}")
print("\n--- Instructions for Local Laptop Serving via Google Drive Sync ---")
print("1. Ensure Google Drive for Desktop is running on your Windows laptop.")
print("2. Start local inference server in watcher mode:")
print('   python inference.py --config configs/imdb_quantitative.yaml --watch-checkpoints "G:\\My Drive\\infinitune\\checkpoints" --device cpu')
print("3. Run training on Colab GPU with drive sync overlay:")
# !python trainer.py --config configs/imdb_quantitative.yaml --config-overlay configs/overlays/drive_sync.yaml --device cuda

### Cell 11: Resume Interrupted Benchmark Run
If the Colab session is preempted or interrupted, resume execution using the identical `--run-dir`.

In [ ]:
# To resume an existing run after reconnect:
!python -m benchmarks.run run \
    --suites streaming_quality \
    --profile standard \
    --model distilgpt2 \
    --live-config configs/imdb_quantitative.yaml \
    --device cuda \
    --kafka-bootstrap localhost:9092 \
    --run-dir {RUN_DIR}

### Cell 12: Archive & Export Benchmark Results
Create a compressed zip archive of the benchmark run directory for export and documentation.

In [ ]:
import shutil

zip_target = PERSIST / f"{RUN_NAME}.zip"
shutil.make_archive(str(zip_target.with_suffix("")), "zip", RUN_DIR)
print(f"Archive written to: {zip_target}")